# Step 1 — Optimize for Behavioral Ground Truth
Hayoung's instructions: build the **group-level behavioral ground truth** from the SocialAHA
character survey, aggregate each Step-0 model's profiles to the **same group level**, then run a
**linear regression per model** to find which sentiment representation best predicts the group emotion ratings.

Step 0 built per-method baselines
Step 1 ranks the baselines against behavior *before* any brain data, so the choice is non-circular
(validate-then-test; Kriegeskorte et al. 2009, Nat. Neurosci. 12:535).

NOTE: The behavioral survey cohort (s1/s2/s3XXX) is a DIFFERENT set of people from the fMRI participants. They share only the 3 scramble GROUPS. So everything here is done at the
**group level**. NOTE TO SELF NEVER merge a behavioral participant id with an fMRI participant id.

## 1.1 Build the behavioral ground truth

**What "ground truth" is here.** A second, model-free measurement of the *same* thing Step 0
measured. In Step 0 we took each fMRI participant's **transcript** and ran it through a sentiment
model to get a valence number per (participant, character, run). Here we take each SONA
participant's **survey answer** — a number they reported directly — for the same (participant,
character, run) grid. Same construct (how positively someone feels about each character at each
run), two routes: one *inferred from speech*, one *self-reported*. Step 1 grades the models against
this answer key. Behavior is the **target**, never a feature.

**Why group-level.** The SONA behavioral cohort (`s1/s2/s3XXX`) are *different people* from the
fMRI cohort; they share only the 3 scramble **groups**. So we can't line participants up one-to-one.
Instead we average to the group level on both sides and compare group trajectories.

**How the averaging works (important).** We group by **(group, Character, Run)** and average across
the ~12–14 participants in that group. **Characters and runs are kept separate** — we do NOT collapse
across characters or across runs. Each character keeps its own 10-run trajectory per group
(e.g. "group 1, Jack, run 3" = the mean of group-1 participants' rating of Jack at run 3). Result:
3 groups × 4 characters × 10 runs = **120 rows**, exactly mirroring the model side in 1.2.

**Inputs / steps**
- One MATLAB file per participant in `data/charsurvey/` (`block1..block10` = Run 1..10; each block is
  16 items × 4 characters), plus `labels.mat` (the 16 item names).
- Group = first digit of the id (`s1/s2/s3` → 1/2/3).
- Character column order is fixed by the survey sheet (instruction.pdf p.6): **Jack, Kate, Randall, Kevin**.
- The single `behavior` target is chosen at the end of the next cell (default = the `positive emotion` item).


---

**Grain & aggregation rationale (for the writeup / Hayoung).**
Step 0 scores one valence per **(participant, run, character)** — the *same unit* the survey uses
(each participant rated each character each run). So the two sides already share a base object; we
did not have to force this, it falls out of the shared 10-run × 4-character paradigm. The **group**
average is added *only here in Step 1*, symmetrically to both sides, purely because the SONA
behavioral cohort and the fMRI cohort are different people and cannot be matched one-to-one. If they
were the same people we would compare at participant level and skip grouping entirely.

Using different grains across the project (group-level here; participant-level within group for the
later brain / IS-RSA step) is fine **as long as both sides of any single comparison are at the same
grain** — which they are. The caveats are about interpretation, not validity: (1) group-level fit
does not license individual-level claims (ecological fallacy / Simpson's paradox, Robinson 1950);
(2) averaging ~12 participants inflates apparent effect sizes, so report these as group-level;
(3) picking the winner at group level and reusing it at participant level in the brain step assumes
the best group-level representation is also the best individual-level one — an explicit assumption,
not a free lunch.


In [1]:
import pandas as pd, numpy as np, scipy.io as sio
from pathlib import Path

CHARACTERS = ["jack", "kate", "kevin", "randall"]

# --- Behavioral data location -------------------------------------------------
# Per-participant survey responses are MATLAB files (one .mat per behavioral
# participant: s1XXX / s2XXX / s3XXX). Drop the charsurvey folder into the repo
# and point BEH_DIR at it. It must also contain labels.mat (the 16 item names).
BEH_DIR    = Path("data/charsurvey")          # <-- folder of s*.mat + labels.mat
LABELS_MAT = BEH_DIR / "labels.mat"

# Column order of each 16x4 block matrix, fixed by the survey sheet
# (instruction.pdf p.6: Jack, Kate, Randall, Kevin -- NOT alphabetical):
CHAR_COLS = ["jack", "kate", "randall", "kevin"]

# The 16 row labels (questionnaire items), read from labels.mat:
ITEM_LABELS = [str(x[0]) for x in sio.loadmat(LABELS_MAT)["labels"].ravel()]

def group_from_id(pid):
    # s1XXX/s2XXX/s3XXX -> 1/2/3  (group = first digit after the 's')
    digits = [c for c in str(pid) if c.isdigit()]
    return int(digits[0]) if digits else np.nan

# STAGE 1 of ground truth: unpack each participant's raw ratings into long form.
# One row per (participant, character, run); the 16 survey items become columns.
# NOTHING is averaged here -- that happens in the next cell.
def load_participant(path):
    pid = Path(path).stem                       # e.g. "s1002"
    m = sio.loadmat(path)
    rows = []
    for b in range(1, 11):                       # block{1..10} == Run 1..10
        blk = m[f"block{b}"]                      # 16 items x 4 characters
        for ci, ch in enumerate(CHAR_COLS):
            rec = {"Participant": pid, "group": group_from_id(pid),
                   "Character": ch, "Run": b}
            for ri, lab in enumerate(ITEM_LABELS):
                # Keep NaN for skipped ratings (s1029, s2027, s3044); group means
                # in 1.1b use available responses only -- we never impute.
                rec[lab] = float(blk[ri, ci])
            rows.append(rec)
    return pd.DataFrame(rows)

# Load every behavioral participant (skip labels.mat / non-subject files)
mat_files = sorted(p for p in BEH_DIR.glob("s*.mat") if p.name != "labels.mat")
assert mat_files, f"No s*.mat files found in {BEH_DIR.resolve()} -- add the charsurvey folder"
beh = pd.concat([load_participant(p) for p in mat_files], ignore_index=True)
beh["Character"] = beh["Character"].str.lower().str.strip()
beh = beh[beh["Character"].isin(CHARACTERS)]

print(f"participants: {beh.Participant.nunique()} | rows: {len(beh)}")
print("per-group N:", beh.groupby('group')['Participant'].nunique().to_dict())
print("items:", ITEM_LABELS)
beh.head()

participants: 39 | rows: 1560
per-group N: {1: 13, 2: 14, 3: 12}
items: ['warm and kind', 'intelligent', 'agreeable', 'extraverted', 'impulsive', 'emotionally stable', 'open-minded', 'trustworthy', 'competent', 'rational behavior', 'positive emotion', 'good relationship', 'empathize', 'understand', 'like', 'similar']


,Participant,group,Character,Run,warm and kind,intelligent,agreeable,extraverted,impulsive,emotionally stable,open-minded,trustworthy,competent,rational behavior,positive emotion,good relationship,empathize,understand,like,similar
0,s1002,1,jack,1,6.0,4.0,6.0,5.0,2.0,6.0,5.0,6.0,6.0,6.0,7.0,7.0,6.0,4.0,6.0,4.0
1,s1002,1,kate,1,6.0,4.0,5.0,6.0,5.0,3.0,4.0,5.0,6.0,5.0,2.0,2.0,5.0,4.0,5.0,4.0
2,s1002,1,randall,1,7.0,6.0,6.0,4.0,5.0,4.0,6.0,6.0,6.0,6.0,3.0,5.0,6.0,4.0,7.0,5.0
3,s1002,1,kevin,1,6.0,4.0,4.0,7.0,6.0,2.0,6.0,5.0,4.0,2.0,2.0,3.0,3.0,4.0,3.0,2.0
4,s1002,1,jack,2,7.0,5.0,6.0,6.0,4.0,6.0,6.0,7.0,6.0,6.0,4.0,7.0,6.0,5.0,7.0,5.0


In [2]:
# STAGE 2 of ground truth: average across PARTICIPANTS within each
# (group, Character, Run) cell. Characters and Runs are the grouping keys, so they
# are KEPT SEPARATE -- we do NOT collapse across characters or across runs. Each
# character keeps its own 10-run trajectory per group. NaNs (skipped ratings) are
# ignored by .mean() (skipna=True), so a cell uses only the participants who answered.
# Result grid: 3 groups x 4 characters x 10 runs = 120 rows -- same shape as the models in 1.2.
gt = (beh.groupby(["group", "Character", "Run"])[ITEM_LABELS]
        .mean()
        .reset_index())

# --- REGRESSION TARGET (open design decision; flagged for Hayoung) ------------
# The research question is about expressed *sentiment/valence*, so the single
# most defensible behavioral target is the dedicated emotion item.
# Switch TARGET_MODE to change what `behavior` is:
#   "positive_emotion" -> the "positive emotion" item alone           (DEFAULT)
#   "last_section"      -> average of empathize/understand/like/similar
#   "custom"           -> average of TARGET_ITEMS below
TARGET_MODE  = "positive_emotion"
TARGET_ITEMS = ["positive emotion"]            # used only when TARGET_MODE=="custom"

if TARGET_MODE == "positive_emotion":
    target_items = ["positive emotion"]
elif TARGET_MODE == "last_section":
    target_items = ["empathize", "understand", "like", "similar"]
elif TARGET_MODE == "custom":
    target_items = TARGET_ITEMS
else:
    raise ValueError(TARGET_MODE)

missing = [c for c in target_items if c not in gt.columns]
assert not missing, f"target items not found: {missing}"
gt["behavior"] = gt[target_items].mean(axis=1)

Path("results/step1").mkdir(parents=True, exist_ok=True)
gt.to_csv("results/step1/01__ground_truth_group_level.csv", index=False)
print(f"target = {TARGET_MODE} {target_items}")
print("ground truth shape:", gt.shape, "| expect ~ 3 groups x 4 chars x 10 runs = 120")
gt[["group","Character","Run","behavior"]].head()

target = positive_emotion ['positive emotion']
ground truth shape: (120, 20) | expect ~ 3 groups x 4 chars x 10 runs = 120


,group,Character,Run,behavior
0,1,jack,1,6.500000
1,1,jack,2,5.230769
2,1,jack,3,4.384615
3,1,jack,4,2.307692
4,1,jack,5,4.076923


## 1.2 Aggregate each Step-0 model to the SAME group level
Each model's profile is per-participant; the ground truth is per-group. To compare, average the
fMRI participants' valence within their group → model trajectory at (group, Character, Run).


In [3]:
# Load each Step-0 per-method baseline and reduce to group-level valence.
#using pos neg baselines
def grp_from_sub(pid):
    digits = [c for c in str(pid) if c.isdigit()]
    return int(digits[0]) if digits else np.nan

MODEL_FILES = {
    "Twitter_RoB": "results/baselines/00__character_vectors_simple_Twitter_RoB.csv",
    "RoBERTa_ZS":  "results/baselines/00__character_vectors_simple_RoBERTa_ZS.csv",
    "VADER":       "results/baselines/00__character_vectors_simple_VADER.csv",
    "Flair":       "results/baselines/00__character_vectors_simple_Flair.csv",
    "SiEBERT":     "results/baselines/00__character_vectors_simple_SiEBERT.csv",
    "BERTweet":    "results/baselines/00__character_vectors_simple_BERTweet.csv",
    "Warriner":    "results/baselines/00b__character_vectors_simple_Warriner_val.csv",   # valence-only, apples-to-apples
    "NRC_VAD":     "results/baselines/00b__character_vectors_simple_NRC_VAD_val.csv",
}

model_group = {}
for name, path in MODEL_FILES.items():
    try:
        d = pd.read_csv(path)
    except FileNotFoundError:
        print(f"[skip {name}: {path} not found]"); continue
    d["Character"] = d["Character"].str.lower().str.strip()
    # valence per reflection: classifiers have positive/negative; lexicons have valence
    if {"positive","negative"}.issubset(d.columns):
        d["valence"] = d["positive"] - d["negative"]
    elif "valence" in d.columns:
        pass
    else:
        print(f"[skip {name}: no valence columns]"); continue
    d["group"] = d["Participant"].map(grp_from_sub)
    # Step-0 left a stray Run == -1 (unassignable reflections; sub-3041 only).
    # Keep only the 10 real runs so the drop is explicit, not an accident of the merge.
    d = d[d["Run"].between(1, 10)]
    g = (d.groupby(["group","Character","Run"])["valence"].mean().reset_index()
           .rename(columns={"valence": f"val_{name}"}))
    model_group[name] = g
    print(f"{name:12s} group-level rows: {len(g)}")

Twitter_RoB  group-level rows: 120
RoBERTa_ZS   group-level rows: 120
VADER        group-level rows: 120
Flair        group-level rows: 120
SiEBERT      group-level rows: 120
BERTweet     group-level rows: 120
Warriner     group-level rows: 120
NRC_VAD      group-level rows: 120


## 1.3 Linear regression per model → which best predicts behavior
For each model, merge its group-level valence with the behavioral ground truth on
(group, Character, Run) and regress. Compare **cross-validated R²** and Spearman.
The best-fitting model becomes the reference baseline (the "winner" of the selection contest).

Methods: standard regression validation (scikit-learn; cross-validated R² to avoid overfitting).
Pick on behavior here, NOT on the brain (Kriegeskorte et al. 2009).

In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from scipy.stats import spearmanr

results = []
for name, g in model_group.items():
    merged = gt.merge(g, on=["group","Character","Run"], how="inner").dropna(
        subset=["behavior", f"val_{name}"])
    if len(merged) < 5:
        print(f"[{name}: too few matched rows ({len(merged)})]"); continue
    X = merged[[f"val_{name}"]].values
    y = merged["behavior"].values
    # cross-validated R^2 (5-fold) + simple Spearman
    r2_cv = cross_val_score(LinearRegression(), X, y, cv=min(5, len(merged)),
                            scoring="r2").mean()
    rho, p = spearmanr(merged[f"val_{name}"], merged["behavior"])
    results.append({"model": name, "n": len(merged),
                    "cv_R2": round(r2_cv, 3), "spearman": round(rho, 3), "p": round(p, 4)})

res = pd.DataFrame(results).sort_values("cv_R2", ascending=False)
print("Which model best predicts group-level behavior (the selection contest):")
print(res.to_string(index=False))
res.to_csv("results/step1/01__step1_model_ranking.csv", index=False)

Which model best predicts group-level behavior (the selection contest):
      model   n  cv_R2  spearman      p
Twitter_RoB 120  0.386     0.597 0.0000
 RoBERTa_ZS 120  0.322     0.549 0.0000
   BERTweet 120  0.305     0.526 0.0000
      VADER 120  0.238     0.459 0.0000
   Warriner 120  0.169     0.436 0.0000
      Flair 120  0.118     0.416 0.0000
    SiEBERT 120  0.055     0.301 0.0008
    NRC_VAD 120  0.008     0.153 0.0943


## Findings — reading the contest (output above)

**Winner: Twitter-RoBERTa** (cv_R² 0.386, Spearman 0.597). A single sentiment number explains ~39%
of the *out-of-sample* variance in the group valence trajectory — strong for a univariate predictor —
and it is also the model that won the external EmoBank benchmark, so external and behavioral evidence
agree. This is now the **reference baseline**: any fuller / fused vector must beat cv_R² 0.386 to matter.

**RoBERTa zero-shot** is a close second (0.322 / 0.549). The two transformers lead, echoing Step 0's
D2 result (transformer pair Spearman 0.84). **VADER** (0.238) is a respectable training-free floor.

**Flair (0.118) and NRC-VAD (0.008) — did our setup handicap them? We tested it; mostly no.**
- *Flair.* Its single signed label forced the "confidence on one pole, 0 on the other" mapping, which
  can't express graded/ambivalent valence. We tried the principled fix — reconstruct P(positive) from
  the saved confidence and rescale to a zero-centred [-1, 1] (scale by distance to the 0.5 neutral
  point) — and it barely moved: **0.118 → 0.120**. Flair's `en-sentiment` is a *binary, no-neutral,
  overconfident* model: scores pile up near 1.0, so even a correct probabilistic remap stays
  near-binary (~4.5% of reflections near zero). Flair's low score is an **architectural ceiling**, not
  our post-processing. Genuinely helping it needs its pre-softmax margin or a 3-class model (a Step-0 change).
- *NRC-VAD.* Not a setup bug either: full coverage (0 NaN), correct sign (Spearman 0.153), and the
  per-model regression fits its own slope/intercept so its 0-1 scale is irrelevant (z-scoring would not
  change it). Its weakness is inherent to **unweighted word-valence averaging** (no negation/intensifier
  handling, compression toward neutral) — exactly why the rule-augmented VADER beat both plain lexicons.

**Bottom line:** the transformers did not win by us hobbling the others — Flair and NRC-VAD hit real
ceilings of their own designs. **Caveats:** group-level fits (~12 participants averaged) run higher
than individual-level would, so conclusions are about group trajectories; the target is the
`positive emotion` item (confirm with Hayoung).

## What this tells us & what's next
- The top row of `results/step1/01__step1_model_ranking.csv` is the **reference baseline** — the sentiment representation
  that best tracks how people actually rated the characters.
- Any **fuller/combined** vector must now beat this on the same metric to justify its complexity.
- Only after this selection do we touch the brain: feed the winning profile into the RSA and ask whether
  its **across-run changes** track **neural changes** (the research question).

Open decisions to ASK HAYOUNG (1) single averaged target vs. 4 separate items;
(2) valence-only vs. full VAD for the lexicons; (3) whether to add a fused/standardized multi-model
vector as an extra 'model' in this same contest.